# 📊 Baseline Model Training & Evaluation

Train **Logistic Regression**, **Decision Tree**, **Random Forest Family** (RF, Extra Trees, Gradient Boosting, AdaBoost, XGBoost), **SVM**, **BiGRU**, and **BiLSTM** on each of the 5 preprocessed molecular embeddings.

**Embeddings:** RDKit Descriptors · Morgan FP · MACCS Keys · Mol2Vec · ChemBERTa

**Metrics (train & test):** Accuracy · Precision (macro) · Specificity (macro) · Sensitivity/Recall (macro) · F1 (macro) · MCC

In [ ]:
# ─── Imports & Configuration ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix
)

# XGBoost
from xgboost import XGBClassifier

# PyTorch (for BiGRU & BiLSTM)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import copy

# ─── Constants ──────────────────────────────────────────────────────────────
LABEL_COLS = ['Sweet', 'Bitter', 'Umami', 'Sour', 'Undefined']
NUM_CLASSES = len(LABEL_COLS)
EMB_DIR    = "./final_embeddings"
SEED       = 42
TEST_SIZE  = 0.2

EMBEDDINGS = {
    "RDKit":     "rdkit_descriptors.csv",
    "Morgan FP": "morgan_fps.csv",
    "MACCS":     "maccs.csv",
    "Mol2Vec":   "mol2vec.csv",
    "ChemBERTa": "chemberta.csv",
}

np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Imports loaded")
print(f"   Embeddings dir: {EMB_DIR}")
print(f"   Test size: {TEST_SIZE}, Seed: {SEED}, Device: {device}")

✅ Imports loaded
   Embeddings dir: ./embeddings
   Test size: 0.2, Seed: 42, Device: cpu


In [2]:
# ─── Helper: Convert multi-label to single-label ───────────────────────────
# Each row has exactly one label=1 (single-label multi-class problem).
# Encode as integer class for sklearn classifiers.

def load_embedding(name):
    """Load embedding CSV, split into X (features) and y (single int label)."""
    path = os.path.join(EMB_DIR, EMBEDDINGS[name])
    df = pd.read_csv(path)
    
    labels_df = df[LABEL_COLS]
    feat_cols = [c for c in df.columns if c not in LABEL_COLS]
    X = df[feat_cols].values
    
    # Convert one-hot labels → single integer class
    # argmax picks the column index with value 1
    y = labels_df.values.argmax(axis=1)
    
    print(f"  {name}: {X.shape[1]} features, {len(y)} samples")
    class_counts = pd.Series(y).map(dict(enumerate(LABEL_COLS))).value_counts()
    for cls, cnt in class_counts.items():
        print(f"    {cls}: {cnt} ({100*cnt/len(y):.1f}%)")
    
    return X, y

# Quick test
print("Loading all embeddings...\n")
data = {}
for name in EMBEDDINGS:
    X, y = load_embedding(name)
    data[name] = (X, y)
    print()

Loading all embeddings...

  RDKit: 141 features, 17821 samples
    Sweet: 10808 (60.6%)
    Bitter: 2844 (16.0%)
    Undefined: 2053 (11.5%)
    Sour: 1540 (8.6%)
    Umami: 576 (3.2%)

  Morgan FP: 1985 features, 17821 samples
    Sweet: 10808 (60.6%)
    Bitter: 2844 (16.0%)
    Undefined: 2053 (11.5%)
    Sour: 1540 (8.6%)
    Umami: 576 (3.2%)

  MACCS: 161 features, 17821 samples
    Sweet: 10808 (60.6%)
    Bitter: 2844 (16.0%)
    Undefined: 2053 (11.5%)
    Sour: 1540 (8.6%)
    Umami: 576 (3.2%)

  Mol2Vec: 300 features, 17821 samples
    Sweet: 10808 (60.6%)
    Bitter: 2844 (16.0%)
    Undefined: 2053 (11.5%)
    Sour: 1540 (8.6%)
    Umami: 576 (3.2%)

  ChemBERTa: 768 features, 17821 samples
    Sweet: 10808 (60.6%)
    Bitter: 2844 (16.0%)
    Undefined: 2053 (11.5%)
    Sour: 1540 (8.6%)
    Umami: 576 (3.2%)



In [3]:
# ─── Metrics: including macro-specificity from confusion matrix ─────────────

def macro_specificity(y_true, y_pred):
    """
    Compute macro-averaged specificity from the confusion matrix.
    Specificity for class i = TN_i / (TN_i + FP_i)
    """
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]
    specificities = []
    for i in range(n_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp       # predicted i but not actually i
        fn = cm[i, :].sum() - tp       # actually i but not predicted i
        tn = cm.sum() - tp - fp - fn   # everything else
        spec_i = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec_i)
    return np.mean(specificities)


def compute_metrics(y_true, y_pred):
    """Compute all 6 metrics for a set of predictions."""
    return {
        "ACC":  accuracy_score(y_true, y_pred),
        "PRE":  precision_score(y_true, y_pred, average="macro", zero_division=0),
        "SPEC": macro_specificity(y_true, y_pred),
        "SENS": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1":   f1_score(y_true, y_pred, average="macro", zero_division=0),
        "MCC":  matthews_corrcoef(y_true, y_pred),
    }

print("✅ Metric functions defined")

✅ Metric functions defined


In [4]:
# ─── Sklearn Model Definitions ───────────────────────────────────────────────

def get_sklearn_models():
    """Return dict of (model_name → (model_instance, needs_scaling))."""
    return {
        # --- Logistic Regression ---
        "Logistic Regression": (
            LogisticRegression(max_iter=2000, solver="lbfgs",
                               random_state=SEED, n_jobs=-1),
            True
        ),
        # --- Decision Tree ---
        "Decision Tree": (
            DecisionTreeClassifier(random_state=SEED),
            False
        ),
        # --- Random Forest Family ---
        "Random Forest": (
            RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
            False
        ),
        "Extra Trees": (
            ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
            False
        ),
        "Gradient Boosting": (
            GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                        max_depth=5, random_state=SEED),
            False
        ),
        "AdaBoost": (
            AdaBoostClassifier(n_estimators=200, learning_rate=0.1,
                                random_state=SEED),
            False
        ),
        "XGBoost": (
            XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                          random_state=SEED, n_jobs=-1, eval_metric="mlogloss",
                          use_label_encoder=False),
            False
        ),
        # --- SVM ---
        "SVM": (
            SVC(kernel="rbf", random_state=SEED, decision_function_shape="ovr"),
            True
        ),
    }

print("✅ Sklearn model definitions ready")
for name, (model, scale) in get_sklearn_models().items():
    print(f"   {name} (scale={scale})")

✅ Sklearn model definitions ready
   Logistic Regression (scale=True)
   Decision Tree (scale=False)
   Random Forest (scale=False)
   Extra Trees (scale=False)
   Gradient Boosting (scale=False)
   AdaBoost (scale=False)
   XGBoost (scale=False)
   SVM (scale=True)


In [5]:
# ─── Sklearn Training Loop ──────────────────────────────────────────────────
results = []

for emb_name in EMBEDDINGS:
    X, y = data[emb_name]
    
    # Identical stratified split for all models on this embedding
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    
    print(f"\n{'='*60}")
    print(f"📦 Embedding: {emb_name}  (train={len(y_train)}, test={len(y_test)})")
    print(f"{'='*60}")
    
    for model_name, (model, needs_scaling) in get_sklearn_models().items():
        print(f"\n  🔧 {model_name} ...", end=" ")
        
        # Optional scaling (fit on train only)
        if needs_scaling:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_train)
            X_te = scaler.transform(X_test)
        else:
            X_tr, X_te = X_train, X_test
        
        # Train
        model.fit(X_tr, y_train)
        
        # Predict on both sets
        y_train_pred = model.predict(X_tr)
        y_test_pred  = model.predict(X_te)
        
        # Compute metrics
        train_metrics = compute_metrics(y_train, y_train_pred)
        test_metrics  = compute_metrics(y_test, y_test_pred)
        
        row = {
            "Model": model_name,
            "Embedding": emb_name,
        }
        for k, v in train_metrics.items():
            row[f"Train_{k}"] = round(v, 4)
        for k, v in test_metrics.items():
            row[f"Test_{k}"] = round(v, 4)
        
        results.append(row)
        print(f"✓  Test F1={test_metrics['F1']:.4f}  Test MCC={test_metrics['MCC']:.4f}")

print(f"\n{'='*60}")
print(f"✅ All {len(results)} sklearn model-embedding combinations trained!")
print(f"{'='*60}")


📦 Embedding: RDKit  (train=14256, test=3565)

  🔧 Logistic Regression ... ✓  Test F1=0.7701  Test MCC=0.6810

  🔧 Decision Tree ... ✓  Test F1=0.7674  Test MCC=0.7178

  🔧 Random Forest ... ✓  Test F1=0.8567  Test MCC=0.8100

  🔧 Extra Trees ... ✓  Test F1=0.8606  Test MCC=0.8115

  🔧 Gradient Boosting ... ✓  Test F1=0.8499  Test MCC=0.8006

  🔧 AdaBoost ... ✓  Test F1=0.4339  Test MCC=0.4069

  🔧 XGBoost ... ✓  Test F1=0.8554  Test MCC=0.8086

  🔧 SVM ... ✓  Test F1=0.8118  Test MCC=0.7500

📦 Embedding: Morgan FP  (train=14256, test=3565)

  🔧 Logistic Regression ... ✓  Test F1=0.7371  Test MCC=0.6722

  🔧 Decision Tree ... ✓  Test F1=0.7912  Test MCC=0.7433

  🔧 Random Forest ... ✓  Test F1=0.8386  Test MCC=0.7922

  🔧 Extra Trees ... ✓  Test F1=0.8446  Test MCC=0.7939

  🔧 Gradient Boosting ... ✓  Test F1=0.8170  Test MCC=0.7652

  🔧 AdaBoost ... ✓  Test F1=0.1908  Test MCC=0.1383

  🔧 XGBoost ... ✓  Test F1=0.8308  Test MCC=0.7760

  🔧 SVM ... ✓  Test F1=0.7848  Test MCC=0.7358

📦

In [6]:
# ─── Sklearn Results Table ───────────────────────────────────────────────────
sklearn_df = pd.DataFrame(results)

col_order = [
    "Model", "Embedding",
    "Train_ACC", "Train_PRE", "Train_SPEC", "Train_SENS", "Train_F1", "Train_MCC",
    "Test_ACC",  "Test_PRE",  "Test_SPEC",  "Test_SENS",  "Test_F1",  "Test_MCC",
]
sklearn_df = sklearn_df[col_order]

print(f"📊 Sklearn results: {len(sklearn_df)} rows\n")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
sklearn_df

📊 Sklearn results: 40 rows



,Model,Embedding,Train_ACC,Train_PRE,Train_SPEC,Train_SENS,Train_F1,Train_MCC,Test_ACC,Test_PRE,Test_SPEC,Test_SENS,Test_F1,Test_MCC
0,Logistic Regression,RDKit,0.8267,0.7972,0.9381,0.7782,0.7866,0.6987,0.8165,0.7841,0.9349,0.7597,0.7701,0.6810
1,Decision Tree,RDKit,0.9897,0.9947,0.9952,0.9824,0.9883,0.9824,0.8365,0.7852,0.9458,0.7525,0.7674,0.7178
2,Random Forest,RDKit,0.9897,0.9877,0.9965,0.9894,0.9885,0.9824,0.8900,0.8686,0.9615,0.8457,0.8567,0.8100
3,Extra Trees,RDKit,0.9897,0.9947,0.9952,0.9824,0.9883,0.9824,0.8912,0.8710,0.9610,0.8512,0.8606,0.8115
4,Gradient Boosting,RDKit,0.9832,0.9833,0.9936,0.9804,0.9818,0.9713,0.8850,0.8658,0.9592,0.8354,0.8499,0.8006
5,AdaBoost,RDKit,0.6815,0.4833,0.8656,0.4169,0.4262,0.3921,0.6878,0.4904,0.8685,0.4257,0.4339,0.4069
6,XGBoost,RDKit,0.9884,0.9864,0.9960,0.9883,0.9873,0.9801,0.8895,0.8702,0.9610,0.8420,0.8554,0.8086
7,SVM,RDKit,0.8850,0.8643,0.9582,0.8419,0.8517,0.8005,0.8567,0.8326,0.9478,0.7952,0.8118,0.7500
8,Logistic Regression,Morgan FP,0.9734,0.9698,0.9906,0.9688,0.9693,0.9545,0.8079,0.7482,0.9388,0.7281,0.7371,0.6722
9,Decision Tree,Morgan FP,0.9842,0.9872,0.9935,0.9740,0.9804,0.9730,0.8508,0.8011,0.9508,0.7822,0.7912,0.7433


## 🧠 Deep Models: BiGRU & BiLSTM

In [7]:
# ─── BiGRU & BiLSTM Model Definitions ───────────────────────────────────────

class SeqDataset(Dataset):
    """Wraps flat features as (batch, seq_len=1, features) for RNN input."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # [N, 1, D]
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class BiRNNClassifier(nn.Module):
    """Bidirectional RNN (GRU or LSTM) classifier."""
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, num_classes=5,
                 dropout=0.3, rnn_type="GRU"):
        super().__init__()
        RNN = nn.GRU if rnn_type == "GRU" else nn.LSTM
        self.rnn = RNN(
            input_size=input_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            bidirectional=True, dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        # x: [B, seq_len=1, D]
        rnn_out, _ = self.rnn(x)          # [B, 1, hidden*2]
        out = rnn_out[:, -1, :]           # last time step [B, hidden*2]
        out = self.dropout(out)
        return self.fc(out)               # [B, num_classes]


def train_deep_model(model, train_loader, val_X, val_y, criterion, optimizer,
                     device, num_epochs=100, patience=15):
    """Train a deep model with early stopping on validation macro F1."""
    best_f1, best_state, no_improve = 0.0, None, 0

    for epoch in range(num_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Validate
        model.eval()
        with torch.no_grad():
            val_logits = model(val_X.to(device))
            val_preds = torch.argmax(val_logits, dim=1).cpu().numpy()
        val_f1 = f1_score(val_y, val_preds, average="macro", zero_division=0)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    if best_state:
        model.load_state_dict(best_state)
    return best_f1

print("✅ BiGRU & BiLSTM definitions ready")

✅ BiGRU & BiLSTM definitions ready


In [8]:
# ─── Deep Model Training Loop ───────────────────────────────────────────────
DEEP_BATCH_SIZE = 64
DEEP_LR = 1e-3
DEEP_EPOCHS = 100
DEEP_PATIENCE = 15
HIDDEN_DIM = 128
NUM_LAYERS = 2
DEEP_DROPOUT = 0.3

deep_results = []

for emb_name in EMBEDDINGS:
    X, y = data[emb_name]
    input_dim = X.shape[1]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )

    # Scale features (always for deep models)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_te = scaler.transform(X_test)

    # Prepare validation split from training data
    X_tr2, X_val, y_tr2, y_val = train_test_split(
        X_tr, y_train, test_size=0.15, random_state=SEED, stratify=y_train
    )

    train_ds = SeqDataset(X_tr2, y_tr2)
    train_loader = DataLoader(train_ds, batch_size=DEEP_BATCH_SIZE, shuffle=True)

    val_X_tensor = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)  # [N, 1, D]
    test_X_tensor = torch.tensor(X_te, dtype=torch.float32).unsqueeze(1)

    # Compute class weights
    counts = np.bincount(y_tr2, minlength=NUM_CLASSES).astype(float)
    cw = torch.tensor(np.sqrt(len(y_tr2) / (NUM_CLASSES * counts)), dtype=torch.float32).to(device)

    print(f"\n{'='*60}")
    print(f"📦 Embedding: {emb_name}  (train={len(y_tr2)}, val={len(y_val)}, test={len(y_test)})")
    print(f"{'='*60}")

    for rnn_type in ["GRU", "LSTM"]:
        model_name = f"Bi{rnn_type}"
        print(f"\n  🔧 {model_name} ...", end=" ")

        model = BiRNNClassifier(
            input_dim, HIDDEN_DIM, NUM_LAYERS, NUM_CLASSES, DEEP_DROPOUT, rnn_type
        ).to(device)

        criterion = nn.CrossEntropyLoss(weight=cw)
        optimizer = optim.AdamW(model.parameters(), lr=DEEP_LR, weight_decay=1e-4)

        best_val_f1 = train_deep_model(
            model, train_loader, val_X_tensor, y_val, criterion, optimizer,
            device, DEEP_EPOCHS, DEEP_PATIENCE
        )

        # Evaluate on full train and test sets
        model.eval()
        with torch.no_grad():
            train_X_full = torch.tensor(X_tr, dtype=torch.float32).unsqueeze(1).to(device)
            test_X_dev = test_X_tensor.to(device)

            y_train_pred = torch.argmax(model(train_X_full), dim=1).cpu().numpy()
            y_test_pred  = torch.argmax(model(test_X_dev), dim=1).cpu().numpy()

        train_metrics = compute_metrics(y_train, y_train_pred)
        test_metrics  = compute_metrics(y_test, y_test_pred)

        row = {"Model": model_name, "Embedding": emb_name}
        for k, v in train_metrics.items():
            row[f"Train_{k}"] = round(v, 4)
        for k, v in test_metrics.items():
            row[f"Test_{k}"] = round(v, 4)

        deep_results.append(row)
        print(f"✓  Val F1={best_val_f1:.4f}  Test F1={test_metrics['F1']:.4f}  Test MCC={test_metrics['MCC']:.4f}")

print(f"\n{'='*60}")
print(f"✅ All {len(deep_results)} deep model-embedding combinations trained!")
print(f"{'='*60}")


📦 Embedding: RDKit  (train=12117, val=2139, test=3565)

  🔧 BiGRU ... ✓  Val F1=0.8473  Test F1=0.8406  Test MCC=0.7912

  🔧 BiLSTM ... ✓  Val F1=0.8443  Test F1=0.8425  Test MCC=0.7909

📦 Embedding: Morgan FP  (train=12117, val=2139, test=3565)

  🔧 BiGRU ... ✓  Val F1=0.8194  Test F1=0.8088  Test MCC=0.7668

  🔧 BiLSTM ... ✓  Val F1=0.8126  Test F1=0.8072  Test MCC=0.7648

📦 Embedding: MACCS  (train=12117, val=2139, test=3565)

  🔧 BiGRU ... ✓  Val F1=0.8305  Test F1=0.8269  Test MCC=0.7763

  🔧 BiLSTM ... ✓  Val F1=0.8295  Test F1=0.8398  Test MCC=0.7860

📦 Embedding: Mol2Vec  (train=12117, val=2139, test=3565)

  🔧 BiGRU ... ✓  Val F1=0.8283  Test F1=0.8174  Test MCC=0.7613

  🔧 BiLSTM ... ✓  Val F1=0.8268  Test F1=0.8210  Test MCC=0.7645

📦 Embedding: ChemBERTa  (train=12117, val=2139, test=3565)

  🔧 BiGRU ... ✓  Val F1=0.8086  Test F1=0.8029  Test MCC=0.7482

  🔧 BiLSTM ... ✓  Val F1=0.8052  Test F1=0.7886  Test MCC=0.7227

✅ All 10 deep model-embedding combinations trained!


## 📋 Combined Results Table

In [9]:
# ─── Combine All Results ─────────────────────────────────────────────────────
deep_df = pd.DataFrame(deep_results)

col_order = [
    "Model", "Embedding",
    "Train_ACC", "Train_PRE", "Train_SPEC", "Train_SENS", "Train_F1", "Train_MCC",
    "Test_ACC",  "Test_PRE",  "Test_SPEC",  "Test_SENS",  "Test_F1",  "Test_MCC",
]
deep_df = deep_df[col_order]

# Combine sklearn + deep model results
results_df = pd.concat([sklearn_df, deep_df], ignore_index=True)

# Display full table
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
results_df

,Model,Embedding,Train_ACC,Train_PRE,Train_SPEC,Train_SENS,Train_F1,Train_MCC,Test_ACC,Test_PRE,Test_SPEC,Test_SENS,Test_F1,Test_MCC
0,Logistic Regression,RDKit,0.8267,0.7972,0.9381,0.7782,0.7866,0.6987,0.8165,0.7841,0.9349,0.7597,0.7701,0.6810
1,Decision Tree,RDKit,0.9897,0.9947,0.9952,0.9824,0.9883,0.9824,0.8365,0.7852,0.9458,0.7525,0.7674,0.7178
2,Random Forest,RDKit,0.9897,0.9877,0.9965,0.9894,0.9885,0.9824,0.8900,0.8686,0.9615,0.8457,0.8567,0.8100
3,Extra Trees,RDKit,0.9897,0.9947,0.9952,0.9824,0.9883,0.9824,0.8912,0.8710,0.9610,0.8512,0.8606,0.8115
4,Gradient Boosting,RDKit,0.9832,0.9833,0.9936,0.9804,0.9818,0.9713,0.8850,0.8658,0.9592,0.8354,0.8499,0.8006
5,AdaBoost,RDKit,0.6815,0.4833,0.8656,0.4169,0.4262,0.3921,0.6878,0.4904,0.8685,0.4257,0.4339,0.4069
6,XGBoost,RDKit,0.9884,0.9864,0.9960,0.9883,0.9873,0.9801,0.8895,0.8702,0.9610,0.8420,0.8554,0.8086
7,SVM,RDKit,0.8850,0.8643,0.9582,0.8419,0.8517,0.8005,0.8567,0.8326,0.9478,0.7952,0.8118,0.7500
8,Logistic Regression,Morgan FP,0.9734,0.9698,0.9906,0.9688,0.9693,0.9545,0.8079,0.7482,0.9388,0.7281,0.7371,0.6722
9,Decision Tree,Morgan FP,0.9842,0.9872,0.9935,0.9740,0.9804,0.9730,0.8508,0.8011,0.9508,0.7822,0.7912,0.7433


In [10]:
results_df.to_csv("results_table.csv", index=False)
